# TP d'optimisation — sujet 2026 (prédiction)

Ce TP contient 2 parties :

1. Optimisation continue : convexe avec contraintes (Uzawa ou pénalisation) et non convexe (algorithme stochastique)

2. Optimisation discrète : un problème d'affectation utilisant l'algorithme du simplexe et un problème de programmation dynamique

La notation du TP tiendra compte

* de la justification de l'existence et unicité de minima globaux pour chaque problème de minimisation rencontré

* d'une explication préalable à chaque cellule et de commentaires après chaque cellule

* de l'illustration des résultats avec des figures

> **Comment ce sujet a été prédit.** On part du TP 2025 (énoncé et corrigé), de l'en-tête 2026 et des 4 notebooks fournis
> (`DescenteGradient`, `Uzawa`, `Pénalisation`, `Sac-a-dos`).
>
> | Bloc | TP 2025 | En-tête 2026 | Prédiction |
> |---|---|---|---|
> | Continu, convexe | Newton sans contrainte, puis Newton sur le système KKT (contrainte d'**égalité**) | « convexe avec contraintes (Uzawa ou pénalisation) » | mêmes $A,k,b,f_1,f_2$ ; contrainte d'**inégalité** convexe ; Uzawa + pénalisation |
> | Continu, non convexe | recuit simulé, `basinhopping`, `dual_annealing` | inchangé | idem, avec en plus le recuit codé à la main |
> | Affectation | `linear_sum_assignment`, `linprog`, `milp`, unicité | inchangé | idem |
> | Programmation dynamique | absent | **nouveau** | sac à dos (le notebook fourni) |
>
> Les fonctions sont écrites de façon **générique** (`J, DJ, F, DF`) : si l'énoncé change la fonction ou les contraintes,
> il suffit de remplacer leur définition.

## Partie 1 — Optimisation continue

### 1.0 Données et fonctions

Soit $A$ une matrice carrée $5\times 5$ et $k,b$ deux vecteurs de $\mathbb{R}^5$. On définit 2 fonctions de $\mathbb{R}^5$ dans $\mathbb{R}$ :

$$f_1(x) = \|x^4\|_1+\vec k\cdot x^3+x\cdot A x - \vec b\cdot x \qquad\text{où } x^n:=(x_1^n,\ldots, x_5^n)$$

$$f_2(x)=f_1(x)+10\,\cos\,(f_1(x))$$

In [ ]:
import numpy as np
import numpy.linalg as nl
import matplotlib.pyplot as plt
import time, itertools, warnings
from scipy import optimize
from scipy.optimize import (minimize, root, basinhopping, dual_annealing, shgo,
                            linprog, milp, LinearConstraint, Bounds,
                            linear_sum_assignment)
warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': 0.3})
rng = np.random.default_rng(0)   # graine fixée : résultats reproductibles
np.random.seed(0)

In [ ]:
k = np.array([1.0, -1.0, 1, -.2, -1])
b = np.array([-1.0, 2.0, -1.5, 2, 1.5])
A = np.array([[ -5, 1, -1, 3, 2],
              [ 1, -5, 1, 1, 2],
              [ 2, 2, -10, 1, -5],
              [ 0, 2, 1, 0, 1],
              [ 1, 3, -2, 0, -2]], dtype=float)

**Question.** Définir $f_1,\ f_2$ (gradients et hessiennes compris) et tracer les graphes $z=f_i(0,\ldots,0,x,0,\ldots,0)$ pour $i=1$ et $2$, où $x$ occupe tour à tour chacune des 5 coordonnées et varie dans $[-4,4]$.

In [ ]:
# À compléter


### Boîte à outils : fonctions des notebooks fournis

In [ ]:
# À compléter


### 1.1 Minimisation convexe sous contraintes d'inégalité : Uzawa et pénalisation

On considère le problème
$$(\mathcal P)\qquad \min_{x\in K} J(x),\qquad K=\{x\in\mathbb{R}^n,\ F(x)\le 0\},\quad F=(F_1,\dots,F_m),$$
et son lagrangien $L(x,p)=J(x)+p\cdot F(x)$ pour $p\in\mathbb{R}_+^m$.

In [ ]:
# À compléter


#### Exemple 1 (dimension 2, pour visualiser)

$$J(x,y)=4x^2+y^2,\qquad F_1(x,y)=1-x-y\le 0,\qquad F_2(x,y)=e^x-y\le 0 .$$

1. Justifier l'existence et l'unicité du minimum.
2. Écrire les conditions de Kuhn–Tucker et en déduire la solution.
3. Appliquer l'algorithme d'Uzawa. Étudier l'influence du pas $\mu$.
4. Appliquer la méthode de pénalisation. Étudier l'influence de $\varepsilon$.
5. Comparer avec `minimize`.

In [ ]:
# À compléter


#### Exemple 1 bis : le test du notebook `Uzawa.ipynb` fourni

$$J(x,y)=4x^2+y^2,\qquad F(x,y)=\big(y^{4/3}e^x-1,\ x-y+1\big)\le 0 .$$

In [ ]:
# À compléter


#### Exemple 2 (données du TP, $\mathbb{R}^5$) : projection sur un convexe

C'est la version **inégalité** de la contrainte du TP 2025. On note
$$U_{ad}=\Big\{x\in\mathbb{R}^5:\ G(x)=x_1^2+x_2^2+x_3^2+x_4^2+x_5^4-1\le 0,\ \ x_1+x_2+x_3+x_4+x_5\le \tfrac12\Big\}$$
et on cherche la projection de $b$ sur $U_{ad}$ :
$$\min_{x\in U_{ad}} J(x)=\tfrac12\|x-b\|^2 .$$

1. Justifier l'existence et l'unicité.
2. Résoudre par Uzawa. Vérifier les conditions KKT.
3. Résoudre par pénalisation.
4. Comparer avec `minimize`.

In [ ]:
# À compléter


### 1.2 Algorithme du recuit simulé : recherche du minimum global de fonctions non convexes

Algorithme :

* Soit $x_k$ donné. Choix de $y$ de façon aléatoire dans un voisinage $U_k$ de $x_k$
* Si $f(y)<f(x_k)$ alors $x_{k+1}=y$
* Sinon
    * $x_{k+1}=y$ avec une proba $p=e^{-(f(y)-f(x_k))/T(k)}$
    * $x_{k+1}=x_k$ avec proba $1-p$

La fonction $T$ est décroissante, constante par paliers, et $U_k$ est un voisinage de plus en plus petit.

1. Tester l'algorithme de Newton pour $f_1$ et $f_2$ avec différentes données initiales.
2. Programmer le recuit simulé et l'appliquer à $f_1$ et $f_2$.
3. Chercher le minimum global de $f_1$ et $f_2$ à l'aide de *shgo*, *basinhopping* ou bien *dual_annealing*. Comparer.

In [ ]:
# À compléter


## Partie 2 — Optimisation discrète

### 2.1 Programmation linéaire entière : problème d'affectation

On souhaite ranger N objets éparpillés sur le sol dans N boîtes sur le sol.

Les boîtes sont **alignées** selon l'axe des x et sont ordonnées de gauche à droite (si $i < i'$, la boîte $i$ est à gauche de la boîte $i'$). La boîte $i$ est centrée sur un point $B_i$ situé sur l'axe des abscisses.
De même l'objet n°$j$ est repéré par le point $O_j$ du plan $(x, y)$. Les objets doivent être rangés de manière à :

* n'avoir qu'un seul objet par boîte,
* ce que tous les objets soient rangés,
* minimiser la distance totale de déplacement, somme des distances nécessaires pour ranger chaque objet.

Pour ranger l'objet n°$j$ dans la boîte n°$i$, la distance nécessaire est $\|O_j - B_i\|$ (distance euclidienne).

Une solution est une matrice binaire $x\in\mathcal{M}_{n}(\{0,1\})$ telle que $x_{i,j} = 1$ si l'objet $j$ est dans la boîte $i$, 0 sinon.

1. Proposer la mise en œuvre la plus simple à l'aide d'une fonction de la librairie optimize et résoudre le problème pour l'exemple donné dans les fichiers de données.

In [ ]:
import os
import pandas as pd

def lire(nom, defaut):
    if os.path.exists(nom):
        return pd.read_csv(nom, sep='\t').values.astype(float)
    print(f"{nom} absent : données intégrées utilisées")
    return defaut

O_def = np.array([[0.093,-0.26],[-0.85,-0.273],[0.873,-1.673],[-0.443,-1.877],[-0.433,0.573],
                  [-1.1,-0.867],[0.867,-0.617],[0.40,-2.117],[-0.967,-0.973],[0.167,0.21],
                  [-1.97,0.48],[-0.74,0.10],[-0.557,0.293],[-0.557,-0.30],[1.117,-1.623]])
B_def = np.c_[-0.5 + np.arange(15)/14, np.zeros(15)]
PositionObjets  = lire('PositionObjets.txt', O_def)
PositionCasiers = lire('PositionCasiers.txt', B_def)
n = PositionCasiers.shape[0]
print("n =", n, "; casiers triés de gauche à droite :", np.all(np.diff(PositionCasiers[:, 0]) > 0))

In [ ]:
# À compléter


2. On veut à présent rajouter la contrainte suivante : l'objet n°1 doit se situer dans la boîte située juste à gauche de la boîte contenant l'objet n°2. On formule le problème comme un programme linéaire en nombres entiers.

a) Comment traduire mathématiquement que la boîte $i$ contient un objet et un seul et que l'objet $j$ se trouve dans une boîte et une seule ?

b) Traduire mathématiquement la nouvelle contrainte de proximité des objets 1 et 2.

c) Implémenter le programme linéaire à l'aide de `scipy.optimize.linprog`. Attention : le coût et $x$ doivent être codés comme des vecteurs et non des matrices.

In [ ]:
# À compléter


3. Proposer une modélisation pour traduire la contrainte que la boîte contenant l'objet n°7 se situe à côté de la boîte contenant l'objet n°5. Modifier le programme pour intégrer cette contrainte supplémentaire.

In [ ]:
# À compléter


4. On aimerait vérifier que la solution obtenue est la seule solution optimale. Proposer une stratégie pour le vérifier et conclure quant à l'unicité de la solution.

In [ ]:
# À compléter


### 2.2 Programmation dynamique : le problème du sac à dos

Étant donné la capacité $W$ d'un sac et des objets de valeurs $v_i$ et de poids $w_i$ (entiers), on cherche la valeur maximale des objets que l'on peut mettre dans le sac sans dépasser la capacité.

Exemple : capacité = 5, objets (valeur, poids) = [(60, 5), (50, 3), (70, 4), (30, 2)], résultat = 80 (les objets de valeurs 50 et 30 tiennent ensemble).

1. Formuler le problème comme un problème d'optimisation en nombres entiers. Justifier l'existence d'une solution. Est-elle unique ?
2. Écrire l'équation de programmation dynamique (Bellman). Programmer l'algorithme et **retrouver les objets choisis**.
3. Vérifier le résultat (force brute pour $n$ petit, `milp`). Comparer avec la relaxation continue (`linprog`).
4. Étudier la complexité.
5. Proposer un moyen de vérifier l'unicité de la solution optimale.

In [ ]:
class Item(object):

    def __init__(self, value, weight):
        self.value = value
        self.weight = weight


def get_maximum_value(items, capacity):
    dp = [0] * (capacity + 1)
    for item in items:
        dp_tmp = [total_value for total_value in dp]
        for current_weight in range(capacity + 1):
            total_weight = current_weight + item.weight
            if total_weight <= capacity:
                dp_tmp[total_weight] = max(dp_tmp[total_weight],
                                           dp[current_weight] + item.value)
        dp = dp_tmp
    return max(dp)


print(get_maximum_value([Item(60, 10), Item(100, 20), Item(120, 30)], 50))
print(get_maximum_value([Item(60, 5), Item(50, 3), Item(70, 4), Item(30, 2)], 5))

In [ ]:
# À compléter
